# Telco Customer Churn Prediction

## Project goal

Explore customer churn patterns, then train classification models to estimate whether a customer may leave. This is a learning project: patterns show associations and do not prove causes.

We will ask:
- Which customer groups have higher churn rates?
- How are contract type, tenure, monthly charges, and services related to churn?
- How well can a model identify customers who churn?


## 1. Load and inspect the data

The dataset contains customer details and a `Churn` target: `Yes` means the customer left; `No` means they stayed. If KaggleHub is not installed in this Python environment, install it once before running the notebook (`%pip install kagglehub`).


In [ ]:
import kagglehub
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Download the dataset and locate its CSV file.
dataset_folder = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
csv_path = next(dataset_folder.glob("*.csv"))
df = pd.read_csv(csv_path)


In [ ]:
# First look: dataset size, example rows, and column types.
print("Rows and columns:", df.shape)
display(df.head())
df.info()


In [ ]:
# Summarize columns and check the overall churn distribution.
display(df.describe(include="all"))
display(df["Churn"].value_counts())


## 2. Clean the data

`TotalCharges` represents money, but it was stored as text. Blank strings are converted to missing values during numeric conversion. In this dataset, those blanks belong to customers with zero tenure, so we treat their total charges as zero. We also check for missing values and duplicates.


In [ ]:
# Count blank TotalCharges entries before converting the column.
blank_total_charges = (df["TotalCharges"].str.strip() == "").sum()
print("Blank TotalCharges entries:", blank_total_charges)

# Convert to numbers; invalid/blank entries become NaN, then fill them with 0.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

print("Missing values by column:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())


## 3. Explore churn patterns (EDA)

A **churn rate** is the number who left divided by all customers in that group, multiplied by 100. Rates help compare groups of different sizes.


### Contract type and churn

Compare the churn rate within each contract group, rather than comparing churn counts alone.


In [ ]:
contract_summary = df.groupby("Contract").agg(
    total_customers=("Churn", "size"),
    churned_customers=("Churn", lambda values: (values == "Yes").sum())
)
contract_summary["churn_rate_percent"] = (
    contract_summary["churned_customers"] / contract_summary["total_customers"] * 100
)
display(contract_summary)

sns.barplot(data=contract_summary.reset_index(), x="Contract", y="churn_rate_percent")
plt.ylabel("Churn rate (%)")
plt.title("Churn rate by contract type")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### Tenure and churn

First compare average tenure for customers who left and stayed. Then group tenure into ranges to see how churn rates change over time.


In [ ]:
tenure_means = df.groupby("Churn")["tenure"].mean()
display(tenure_means)

df["tenuregroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, df["tenure"].max()],
    labels=["0-12", "13-24", "25+"]
)

tenure_summary = df.groupby("tenuregroup", observed=False).agg(
    total_customers=("Churn", "size"),
    churned_customers=("Churn", lambda values: (values == "Yes").sum())
)
tenure_summary["churn_rate_percent"] = (
    tenure_summary["churned_customers"] / tenure_summary["total_customers"] * 100
)
display(tenure_summary)

plt.bar(tenure_summary.index.astype(str), tenure_summary["churn_rate_percent"])
plt.xlabel("Tenure group (months)")
plt.ylabel("Churn rate (%)")
plt.title("Churn rate by tenure group")
plt.show()


### Monthly charges and churn

Compare average charges and view the spread for customers who stayed versus those who churned. A box plot shows the median and spread, not just the average.


In [ ]:
display(df.groupby("Churn")["MonthlyCharges"].mean())

sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly charges by churn status")
plt.show()


### Internet service, tech support, and payment method

For each category, calculate the percentage of customers who churned. We calculate group summaries directly; we do not add churn-derived rates back into the customer features used for modeling.


In [ ]:
def churn_rate_summary(column):
    summary = df.groupby(column).agg(
        churn_rate_percent=("Churn", lambda values: (values == "Yes").mean() * 100)
    )
    return summary

internet_summary = churn_rate_summary("InternetService")
tech_support_summary = churn_rate_summary("TechSupport")
payment_summary = churn_rate_summary("PaymentMethod")

display(internet_summary)
display(tech_support_summary)
display(payment_summary)


In [ ]:
# Plot each category's churn rate.
for column, summary in [
    ("InternetService", internet_summary),
    ("TechSupport", tech_support_summary),
    ("PaymentMethod", payment_summary),
]:
    sns.barplot(data=summary.reset_index(), x=column, y="churn_rate_percent")
    plt.ylabel("Churn rate (%)")
    plt.title(f"Churn rate by {column}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 4. Prepare the data for machine learning

We want the model to predict `Churn` from customer details. `y` is the answer; `X` is the input information. We remove `customerID` because it is only an identifier. The tenure group is retained as an engineered feature for this learning exercise.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# y contains the answers; X contains the clues.
y = df["Churn"]
X = df.drop(columns=["Churn", "customerID"])

# Keep 20% aside for a final check; preserve the Yes/No proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_columns = X_train.select_dtypes(include=["object", "category"]).columns
numeric_columns = X_train.select_dtypes(include=["number"]).columns

# Text categories become indicator columns; numeric columns are scaled.
preprocessor = ColumnTransformer(transformers=[
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ("numbers", StandardScaler(), numeric_columns)
])


## 5. Train a baseline classification model

Logistic Regression is a classifier here: it predicts `Yes` or `No`. The pipeline first prepares the columns, then trains the classifier. Keeping these steps together ensures the same preparation is used later for new data.


In [ ]:
model = Pipeline(steps=[
    ("preparation", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


### Evaluate the baseline

The test set contains customers the model did not learn from. Accuracy is the share of all predictions that are correct, but the confusion matrix and churn recall help show whether the model finds customers who actually left.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

baseline_predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print(classification_report(y_test, baseline_predictions))
print("Confusion matrix (actual rows, predicted columns; No then Yes):")
print(confusion_matrix(y_test, baseline_predictions, labels=["No", "Yes"]))


### Cross-validation and class imbalance

Cross-validation repeats training and checking on several parts of the training set. We focus on churn recall (the share of actual churners found) and precision (the share of predicted churners who really churned). The test set remains untouched during model comparison.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, recall_score, precision_score
from sklearn.base import clone

churn_recall = make_scorer(recall_score, pos_label="Yes")
churn_precision = make_scorer(precision_score, pos_label="Yes")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_recall_scores = cross_val_score(
    model, X_train, y_train, cv=cv, scoring=churn_recall
)
baseline_precision_scores = cross_val_score(
    model, X_train, y_train, cv=cv, scoring=churn_precision
)

balanced_model = clone(model).set_params(classifier__class_weight="balanced")
balanced_recall_scores = cross_val_score(
    balanced_model, X_train, y_train, cv=cv, scoring=churn_recall
)
balanced_precision_scores = cross_val_score(
    balanced_model, X_train, y_train, cv=cv, scoring=churn_precision
)


`class_weight="balanced"` asks the classifier to pay more attention to the less common churn class. It often increases recall while reducing precision, so compare both scores.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_model = Pipeline(steps=[
    ("preparation", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

forest_recall_scores = cross_val_score(
    forest_model, X_train, y_train, cv=cv, scoring=churn_recall
)
forest_precision_scores = cross_val_score(
    forest_model, X_train, y_train, cv=cv, scoring=churn_precision
)

model_comparison = pd.DataFrame({
    "churn_recall": [baseline_recall_scores.mean(), balanced_recall_scores.mean(), forest_recall_scores.mean()],
    "churn_precision": [baseline_precision_scores.mean(), balanced_precision_scores.mean(), forest_precision_scores.mean()]
}, index=["Logistic Regression", "Balanced Logistic Regression", "Balanced Random Forest"])
display(model_comparison)


## 6. Tune Logistic Regression

Hyperparameter tuning tries settings and selects the one with the strongest cross-validation score. Here, `C` controls the strength of regularization. We use F1 for the churn class to balance precision and recall.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

churn_f1 = make_scorer(f1_score, pos_label="Yes")
parameter_options = {"classifier__C": [0.01, 0.1, 1, 10]}

tuning = GridSearchCV(
    estimator=balanced_model,
    param_grid=parameter_options,
    scoring=churn_f1,
    cv=cv
)
tuning.fit(X_train, y_train)

print("Best settings:", tuning.best_params_)
print("Best cross-validation F1:", tuning.best_score_)


## 7. Final test-set evaluation

Now evaluate the tuned model once on the held-out test customers. This is the final estimate of how it performs on unseen examples; avoid using this result to keep changing settings.


In [ ]:
best_model = tuning.best_estimator_
final_predictions = best_model.predict(X_test)

print(classification_report(y_test, final_predictions))
print("Confusion matrix (actual rows, predicted columns; No then Yes):")
print(confusion_matrix(y_test, final_predictions, labels=["No", "Yes"]))


## 8. Understand the model's signals

Logistic Regression coefficients show which prepared features push the model toward a churn or stay prediction, while considering other features. They describe associations in this model, not causes.


In [ ]:
preparation = best_model.named_steps["preparation"]
classifier = best_model.named_steps["classifier"]
feature_names = preparation.get_feature_names_out()
coefficients = classifier.coef_[0]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print("Features pushing toward a churn prediction:")
display(coefficient_table.sort_values("coefficient", ascending=False).head(10))
print("Features pushing toward a stay prediction:")
display(coefficient_table.sort_values("coefficient", ascending=True).head(10))


## 9. Findings and business takeaways

### EDA findings
- Month-to-month customers had the highest contract churn rate (about 42.7%); two-year customers had the lowest (about 2.8%).
- Churn rates declined across tenure groups: about 47.4% for 0–12 months, 28.7% for 13–24 months, and 14.0% for 25+ months.
- Fiber optic customers had a higher churn rate (about 41.9%) than DSL (19.0%) or customers with no internet service (7.4%).
- Customers without tech support had higher churn than those with support; electronic check users had the highest payment-method churn rate (about 45.3%).

### Model finding
The tuned model caught about 79% of churners in the test set, with about 51% precision. It found many customers who left, while also flagging some who stayed.

### Possible business action
Consider a low-cost early check-in for newer month-to-month customers, and investigate service or billing issues among groups with higher churn. Pilot retention actions and measure results before scaling them.

### Limitations
These patterns do not prove causes. The model has false alarms, and the business should weigh the cost of outreach against the value of catching more potential churners.


## 10. Conclusion

This project followed a supervised-learning workflow: inspect and clean data, explore churn patterns, prepare features, train and compare classifiers, use cross-validation and tuning, evaluate on held-out data, and interpret model signals. Further work could test more models, evaluate other decision thresholds, and package the notebook with a README and project files for GitHub.
